In [1]:
import nest_asyncio
nest_asyncio.apply()
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import RunReportRequest, DateRange, Dimension, Metric
import pandas as pd
from datetime import timedelta,datetime
import os
import numpy as np 

In [2]:
PROPERTY_ID = "properties/220996928"  # replace with your actual GA4 property ID
SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

In [3]:
# command for assigning token 
def cred_save():
    cred=None
    if os.path.exists("token.json"):
        cred=Credentials.from_authorized_user_file("token.json",SCOPES)
    if not cred or not cred.valid:
        if cred and cred.expired and cred.refresh_token:
            cred.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("client_secret_luvkey.json", SCOPES)
            cred=flow.run_local_server(port=0)
        with open("token.json","w") as token:
            token.write(cred.to_json())
    return cred
    

In [4]:
credentials=cred_save()
client= BetaAnalyticsDataClient(credentials=credentials)

In [5]:
# creating the date  variables 
today=datetime.now().date()
yesterday=today-timedelta(days=1)
start=yesterday-timedelta(days=30)

In [6]:
#  Active Users for yesterday 

request = RunReportRequest(
    property=PROPERTY_ID,
    dimensions=[Dimension(name="date")],
    metrics=[Metric(name="activeUsers")],
    date_ranges=[DateRange(start_date=str(start), end_date=str(yesterday))],
)
response = client.run_report(request)

In [7]:
"""# --- PRINT RESULT ---
print("Raw response rows:")

for row in response.rows:
    date = row.dimension_values[0].value
    active_users = row.metric_values[0].value
    print(f"Date: {date} | Active Users: {active_users}")


if not response.rows:
    print("No data returned — check property ID, date range, or permissions.")
"""

'# --- PRINT RESULT ---\nprint("Raw response rows:")\n\nfor row in response.rows:\n    date = row.dimension_values[0].value\n    active_users = row.metric_values[0].value\n    print(f"Date: {date} | Active Users: {active_users}")\n\n\nif not response.rows:\n    print("No data returned — check property ID, date range, or permissions.")\n'

In [8]:
d=[]

for row in response.rows:
    dimension_val=row.dimension_values[0].value
    metric_val=row.metric_values[0].value
    d.append({"date":dimension_val,"average user daily":metric_val})



In [9]:
df=pd.DataFrame(d)

In [10]:
df['date']=pd.to_datetime(df['date'],format="%Y%m%d")

In [11]:

df=df.sort_values(by='date',ascending=True)
df.reset_index(drop=True)


,date,average user daily
0,2026-05-25,69914
1,2026-05-26,68754
2,2026-05-27,70323
3,2026-05-28,68239
4,2026-05-29,69309
5,2026-05-30,72361
6,2026-05-31,75177
7,2026-06-01,72371
8,2026-06-02,72423
9,2026-06-03,73553


In [12]:
df['average user daily']=df['average user daily'].astype(int)

In [16]:
y=df["average user daily"].values
standard_dev=np.std(y)
standard_dev# 
mean_user=np.mean(y)
yesterday_val=df[30:]["average user daily"]

In [14]:
mean_user

np.float64(70904.06451612903)

In [ ]:
# I will be using asymmetric anamoly detection  
if yesterday_val>mean_user:
    

SyntaxError: invalid syntax (3626318106.py, line 2)